# LongMemEval Experiment Kaggle Pipeline

This notebook runs a resource-light LongMemEval reproduction pipeline on Kaggle:

1. Prepare the LongMemEval source tree.
2. Download or import the cleaned benchmark data.
3. Run memory retrieval with BM25 at session granularity.
4. Run retrieval-augmented generation through an OpenAI-compatible API.
5. Run the official LLM-as-judge QA evaluator and summarize results.

The default run is a smoke test: 20 examples, CPU BM25 retrieval, and API-based generation/evaluation. Set `RUN_FULL=1` or edit `RUN_FULL = True` to evaluate all 500 examples.

## Kaggle requirements

- Internet must be enabled unless you attach the benchmark JSON files as a Kaggle Dataset.
- Add `OPENAI_API_KEY` in Kaggle Add-ons -> Secrets.
- If you use a non-OpenAI router, also add `OPENAI_BASE_URL` as a Kaggle Secret or set it in the configuration cell.
- The notebook never prints the API key and does not pass it as a command-line argument.


In [1]:
from pathlib import Path
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import urllib.request

ROOT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
PROJECT_DIR_NAME = os.getenv('PROJECT_DIR_NAME', 'LongMemEval-Experiment')
REPO_URL = os.getenv('LONGMEMEVAL_REPO', 'https://github.com/toanthangO20/LongMemEval-Experiment.git')
CHECKOUT_DIR = os.getenv('LONGMEMEVAL_CHECKOUT_DIR', PROJECT_DIR_NAME)

# Choose the benchmark file. Use longmemeval_oracle.json for the cheapest generation sanity checks.
DATASET_NAME = os.getenv('LONGMEMEVAL_DATASET', 'longmemeval_s_cleaned.json')

# Smoke-test defaults. For full reproduction, set RUN_FULL=True or RUN_FULL=1 and use all 500 examples.
RUN_FULL = os.getenv('RUN_FULL', '0') == '1'
N_EXAMPLES = int(os.getenv('N_EXAMPLES', '20'))
RANDOM_SEED = int(os.getenv('RANDOM_SEED', '7'))

# Retrieval configuration for the resource-light baseline.
RETRIEVER = os.getenv('RETRIEVER', 'flat-bm25')
GRANULARITY = os.getenv('GRANULARITY', 'session')
TOPK_CONTEXT = int(os.getenv('TOPK_CONTEXT', '50'))

# OpenAI-compatible reader and evaluator configuration.
# Defaults use OpenAI directly. For routers, set OPENAI_BASE_URL and custom model names below.
GEN_MODEL_NAME = os.getenv('GEN_MODEL_NAME', 'cx/gpt-5.2')
GEN_MODEL_ALIAS = os.getenv('GEN_MODEL_ALIAS', 'router-gpt-5.2')
METRIC_MODEL_SHORT = os.getenv('METRIC_MODEL_SHORT', 'router-gpt-5.2')
METRIC_MODEL_NAME = os.getenv('METRIC_MODEL_NAME', 'cx/gpt-5.2')
MODEL_MAX_LENGTH = int(os.getenv('MODEL_MAX_LENGTH', '128000'))
HISTORY_FORMAT = os.getenv('HISTORY_FORMAT', 'json')
USERONLY = os.getenv('USERONLY', 'false')
OPENAI_DEFAULT_HEADERS = os.getenv('OPENAI_DEFAULT_HEADERS', '{}')

print('Working root:', ROOT)
print('Dataset:', DATASET_NAME)
print('Run full benchmark:', RUN_FULL, '| N_EXAMPLES:', N_EXAMPLES)
print('Retriever:', RETRIEVER, '| Granularity:', GRANULARITY)
print('Generation model:', GEN_MODEL_NAME)
print('Metric model alias:', METRIC_MODEL_SHORT, '| metric model:', METRIC_MODEL_NAME)


Working root: /kaggle/working
Dataset: longmemeval_s_cleaned.json
Run full benchmark: False | N_EXAMPLES: 20
Retriever: flat-bm25 | Granularity: session
Generation model: cx/gpt-5.2
Metric model alias: router-gpt-5.2 | metric model: cx/gpt-5.2


## Install lightweight dependencies

The full project requirements include vLLM and heavier CUDA packages. For this BM25 + API pipeline, these packages are enough. `httpx==0.27.2` is pinned for compatibility with `openai==1.35.1`.


In [2]:
%pip install -q openai==1.35.1 httpx==0.27.2 backoff==2.2.1 rank-bm25==0.2.2 tiktoken==0.7.0 sentence-transformers==2.7.0 scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.8/326.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 77.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not inst

In [3]:
import httpx
import openai
print('openai version:', openai.__version__)
print('httpx version:', httpx.__version__)
assert tuple(map(int, httpx.__version__.split('.')[:2])) < (0, 28), 'httpx must be < 0.28 for openai==1.35.1'


openai version: 1.35.1
httpx version: 0.27.2


## Load secrets and helper functions

Secrets are read from environment variables first, then from Kaggle Secrets. The API key is only stored in the process environment and is not passed to subprocess command lines.


In [4]:
def run_cmd(cmd, cwd=None, env=None, check=True):
    shown = [str(x) for x in cmd]
    print('$', ' '.join(shown))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, check=check)


def load_secret(name):
    value = os.getenv(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ''


OPENAI_API_KEY = load_secret('OPENAI_API_KEY')
OPENAI_ORGANIZATION = load_secret('OPENAI_ORGANIZATION')
OPENAI_BASE_URL = load_secret('OPENAI_BASE_URL') or os.getenv('OPENAI_BASE_URL', '')

if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
if OPENAI_ORGANIZATION:
    os.environ['OPENAI_ORGANIZATION'] = OPENAI_ORGANIZATION
if OPENAI_BASE_URL:
    os.environ['OPENAI_BASE_URL'] = OPENAI_BASE_URL
os.environ['OPENAI_DEFAULT_HEADERS'] = OPENAI_DEFAULT_HEADERS
os.environ['TOKENIZER_BACKEND'] = os.getenv('TOKENIZER_BACKEND', 'openai')
os.environ['MODEL_MAX_LENGTH'] = str(MODEL_MAX_LENGTH)
os.environ['METRIC_MODEL_NAME'] = METRIC_MODEL_NAME

print('OPENAI_API_KEY configured:', bool(OPENAI_API_KEY))
print('OPENAI_ORGANIZATION configured:', bool(OPENAI_ORGANIZATION))
print('OPENAI_BASE_URL configured:', bool(OPENAI_BASE_URL))
print('MODEL_MAX_LENGTH:', MODEL_MAX_LENGTH)
print('TOKENIZER_BACKEND:', os.environ['TOKENIZER_BACKEND'])


OPENAI_API_KEY configured: True
OPENAI_ORGANIZATION configured: False
OPENAI_BASE_URL configured: True
MODEL_MAX_LENGTH: 128000
TOKENIZER_BACKEND: openai


## Prepare the source tree

When the notebook is run from a repository checkout, it uses that checkout. Otherwise it clones this public experiment repository into `/kaggle/working`. Set `LONGMEMEVAL_REPO` only if you intentionally want to test another fork.


In [5]:
def looks_like_longmemeval_repo(path):
    path = Path(path)
    return (path / 'src' / 'retrieval' / 'run_retrieval.py').exists() and (path / 'src' / 'generation' / 'run_generation.py').exists()


candidate_dirs = [Path.cwd(), ROOT / CHECKOUT_DIR, ROOT / 'LongMemEval', ROOT / PROJECT_DIR_NAME]
REPO_DIR = None
for candidate in candidate_dirs:
    if looks_like_longmemeval_repo(candidate):
        REPO_DIR = candidate.resolve()
        break

if REPO_DIR is None:
    REPO_DIR = (ROOT / CHECKOUT_DIR).resolve()
    if not REPO_DIR.exists():
        run_cmd(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
    if not looks_like_longmemeval_repo(REPO_DIR):
        raise RuntimeError(f'Checkout does not look like a LongMemEval repo: {REPO_DIR}')

print('Using source tree:', REPO_DIR)
print('Top-level files:')
for path in sorted(REPO_DIR.iterdir()):
    if path.name != '.git':
        print(' -', path.name)


$ git clone --depth 1 https://github.com/toanthangO20/LongMemEval-Experiment.git /kaggle/working/LongMemEval-Experiment


Cloning into '/kaggle/working/LongMemEval-Experiment'...


Using source tree: /kaggle/working/LongMemEval-Experiment
Top-level files:
 - .gitignore
 - LICENSE
 - README.md
 - assets
 - data
 - longmemeval-kaggle-reproduce-pipeline.ipynb
 - requirements-full.txt
 - requirements-lite.txt
 - src


## Apply Kaggle compatibility patches

These patches are idempotent. They keep the benchmark logic intact while making the scripts safer for Kaggle: no API key in printed args, optional OpenAI-compatible base URL, custom metric model alias support, and NumPy 2.x compatibility for retrieval metrics.


In [6]:
def replace_text(path, old, new):
    text = path.read_text(encoding='utf-8')
    if old in text:
        path.write_text(text.replace(old, new), encoding='utf-8')
        return True
    return False


gen_py = REPO_DIR / 'src' / 'generation' / 'run_generation.py'
gen_text = gen_py.read_text(encoding='utf-8')
if 'import os\n' not in gen_text[:150]:
    gen_text = gen_text.replace('import sys\n', 'import sys\nimport os\n')
gen_py.write_text(gen_text, encoding='utf-8')

replace_text(
    gen_py,
    "    if args.openai_organization:\n        openai.organization = args.openai_organization\n",
    "    openai_organization = args.openai_organization or os.getenv('OPENAI_ORGANIZATION')\n    if openai_organization:\n        openai.organization = openai_organization\n",
)

replace_text(
    gen_py,
    "    parser.add_argument('--openai_key', type=str, required=True)\n",
    "    parser.add_argument('--openai_key', type=str, default=None)\n",
)
replace_text(
    gen_py,
    "def check_args(args):\n    print(args)\n",
    "def check_args(args):\n    safe_args = argparse.Namespace(**vars(args))\n    if safe_args.openai_key:\n        safe_args.openai_key = '***'\n    if safe_args.openai_organization:\n        safe_args.openai_organization = '***'\n    print(safe_args)\n",
)
replace_text(
    gen_py,
    "    client = OpenAI(\n        api_key=args.openai_key,\n        base_url=args.openai_base_url,\n    )",
    "    openai_key = args.openai_key or os.getenv('OPENAI_API_KEY')\n    if not openai_key:\n        raise RuntimeError('OPENAI_API_KEY is required. Set it in the environment or pass --openai_key.')\n    default_headers = json.loads(os.getenv('OPENAI_DEFAULT_HEADERS', '{}'))\n    client = OpenAI(\n        api_key=openai_key,\n        base_url=args.openai_base_url,\n        default_headers=default_headers,\n    )",
)
replace_text(
    gen_py,
    "    model_max_length = model2maxlength[args.model_name]\n",
    "    model_max_length = model2maxlength.get(args.model_name, int(os.getenv('MODEL_MAX_LENGTH', '128000')))\n",
)
replace_text(
    gen_py,
    "    if 'gpt-4' in args.model_name.lower()  or 'gpt-3.5' in args.model_name.lower():\n        tokenizer = tiktoken.get_encoding('o200k_base')\n        tokenizer_backend = 'openai'\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(args.model_name)\n        tokenizer_backend = 'huggingface'\n",
    "    if os.getenv('TOKENIZER_BACKEND', '').lower() == 'openai' or 'gpt-4' in args.model_name.lower()  or 'gpt-3.5' in args.model_name.lower():\n        tokenizer = tiktoken.get_encoding('o200k_base')\n        tokenizer_backend = 'openai'\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(args.model_name)\n        tokenizer_backend = 'huggingface'\n",
)
replace_text(
    gen_py,
    "            total_prompt_tokens += completion.usage.prompt_tokens\n            total_completion_tokens += completion.usage.completion_tokens\n",
    "            usage = getattr(completion, 'usage', None)\n            total_prompt_tokens += (getattr(usage, 'prompt_tokens', 0) or 0)\n            total_completion_tokens += (getattr(usage, 'completion_tokens', 0) or 0)\n",
)

eval_py = REPO_DIR / 'src' / 'evaluation' / 'evaluate_qa.py'
if "METRIC_MODEL_NAME" not in eval_py.read_text(encoding='utf-8'):
    replace_text(
        eval_py,
        "    if metric_model_short not in model_zoo:\n        print('Requested metric model is not supported:', metric_model_short)\n        exit()\n",
        "    if metric_model_short not in model_zoo and os.getenv('METRIC_MODEL_NAME'):\n        model_zoo[metric_model_short] = (os.getenv('METRIC_MODEL_NAME'), 'openai')\n    if metric_model_short not in model_zoo:\n        print('Requested metric model is not supported:', metric_model_short)\n        exit()\n",
    )
replace_text(
    eval_py,
    "        openai_api_base = None\n",
    "        openai_api_base = os.getenv('OPENAI_BASE_URL') or None\n        if not openai_api_key:\n            raise RuntimeError('OPENAI_API_KEY is required for OpenAI-compatible evaluation models.')\n",
)
replace_text(
    eval_py,
    "    metric_client = OpenAI(\n        api_key=openai_api_key,\n        base_url=openai_api_base,\n    )",
    "    default_headers = json.loads(os.getenv('OPENAI_DEFAULT_HEADERS', '{}'))\n    metric_client = OpenAI(\n        api_key=openai_api_key,\n        base_url=openai_api_base,\n        default_headers=default_headers,\n    )",
)

eval_utils_py = REPO_DIR / 'src' / 'retrieval' / 'eval_utils.py'
replace_text(eval_utils_py, 'np.asfarray(relevances)[:k]', 'np.asarray(relevances, dtype=float)[:k]')

print('Compatibility patches applied or already present.')


Compatibility patches applied or already present.


## Fetch benchmark data

The notebook first searches `/kaggle/input` for `DATASET_NAME`. If the file is not attached as a Kaggle Dataset, it downloads the official cleaned benchmark file from Hugging Face.


In [7]:
DATA_DIR = REPO_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
target_file = DATA_DIR / DATASET_NAME

if not target_file.exists():
    candidates = list(Path('/kaggle/input').rglob(DATASET_NAME)) if Path('/kaggle/input').exists() else []
    if candidates:
        print('Copying dataset from Kaggle input:', candidates[0])
        shutil.copy2(candidates[0], target_file)
    else:
        url = f'https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/{DATASET_NAME}'
        print('Downloading:', url)
        urllib.request.urlretrieve(url, target_file)
else:
    print('Dataset already exists:', target_file)

data = json.loads(target_file.read_text(encoding='utf-8'))
print('Loaded examples:', len(data))
print('First example keys:', sorted(data[0].keys()))
counts = {}
for row in data:
    counts[row['question_type']] = counts.get(row['question_type'], 0) + 1
print('Question type counts:')
print(json.dumps(counts, indent=2))


Downloading: https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json
Loaded examples: 500
First example keys: ['answer', 'answer_session_ids', 'haystack_dates', 'haystack_session_ids', 'haystack_sessions', 'question', 'question_date', 'question_id', 'question_type']
Question type counts:
{
  "single-session-user": 70,
  "multi-session": 133,
  "single-session-preference": 30,
  "temporal-reasoning": 133,
  "knowledge-update": 78,
  "single-session-assistant": 56
}


## Build a reproducible sample file

A sample keeps API spend controlled. To reproduce full benchmark metrics, set `RUN_FULL=True` in the configuration cell or define Kaggle environment variable `RUN_FULL=1`.


In [8]:
if RUN_FULL:
    work_file = target_file
    work_data = data
else:
    rng = random.Random(RANDOM_SEED)
    work_data = data.copy()
    rng.shuffle(work_data)
    work_data = work_data[:N_EXAMPLES]
    sample_name = f'{target_file.stem}_sample{len(work_data)}_seed{RANDOM_SEED}.json'
    work_file = DATA_DIR / sample_name
    work_file.write_text(json.dumps(work_data, ensure_ascii=False), encoding='utf-8')

print('Active benchmark file:', work_file)
print('Active examples:', len(work_data))


Active benchmark file: /kaggle/working/LongMemEval-Experiment/data/longmemeval_s_cleaned_sample20_seed7.json
Active examples: 20


## Step 1: memory retrieval

This runs the repository retrieval code and writes a JSONL retrieval log containing `retrieval_results`. The default is `flat-bm25` over sessions, which is CPU-friendly and mirrors the baseline memory retrieval stage.


In [9]:
retrieval_out_dir = REPO_DIR / 'retrieval_logs' / RETRIEVER / GRANULARITY
retrieval_out_dir.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR) + os.pathsep + env.get('PYTHONPATH', '')

cmd = [
    sys.executable, 'run_retrieval.py',
    '--in_file', str(work_file),
    '--retriever', RETRIEVER,
    '--granularity', GRANULARITY,
    '--index_expansion_method', 'none',
    '--index_expansion_result_join_mode', 'none',
    '--index_expansion_result_cache', 'none',
    '--out_dir', str(retrieval_out_dir),
    '--outfile_prefix', work_file.name,
    '--cache_dir', str(REPO_DIR / 'model_cache'),
]
run_cmd(cmd, cwd=REPO_DIR / 'src' / 'retrieval', env=env)

retrieval_log = retrieval_out_dir / f'{work_file.name}_retrievallog_{GRANULARITY}_{RETRIEVER}'
print('Retrieval log:', retrieval_log)
print('Exists:', retrieval_log.exists(), '| Size MB:', round(retrieval_log.stat().st_size / 1e6, 2) if retrieval_log.exists() else None)


$ /usr/bin/python3 run_retrieval.py --in_file /kaggle/working/LongMemEval-Experiment/data/longmemeval_s_cleaned_sample20_seed7.json --retriever flat-bm25 --granularity session --index_expansion_method none --index_expansion_result_join_mode none --index_expansion_result_cache none --out_dir /kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session --outfile_prefix longmemeval_s_cleaned_sample20_seed7.json --cache_dir /kaggle/working/LongMemEval-Experiment/model_cache
Namespace(in_file='/kaggle/working/LongMemEval-Experiment/data/longmemeval_s_cleaned_sample20_seed7.json', out_dir='/kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session', outfile_prefix='longmemeval_s_cleaned_sample20_seed7.json', cache_dir='/kaggle/working/LongMemEval-Experiment/model_cache', retriever='flat-bm25', granularity='session', index_expansion_method='none', index_expansion_llm=None, index_expansion_result_cache='none', index_expansion_result_join_mode='none')
Setting num process

100%|██████████| 2/2 [00:00<00:00, 86.68it/s]


Ignored 1 instances due to abstention: {'gpt4_372c3eed_abs'}
Additionally ignored 5 instances due to no target turns from the user side: {'51b23612', '16c90bf4', 'ac031881', '1568498a', '8752c811'}
{"session": {"recall_any@1": 0.8571428571428571, "recall_all@1": 0.14285714285714285, "ndcg_any@1": 0.8571428571428571, "recall_any@3": 1.0, "recall_all@3": 0.6428571428571429, "ndcg_any@3": 0.8368524078832428, "recall_any@5": 1.0, "recall_all@5": 0.9285714285714286, "ndcg_any@5": 0.8943444481005786, "recall_any@10": 1.0, "recall_all@10": 1.0, "ndcg_any@10": 0.9115671300423678, "recall_any@30": 1.0, "recall_all@30": 1.0, "ndcg_any@30": 0.9115671300423678, "recall_any@50": 1.0, "recall_all@50": 1.0, "ndcg_any@50": 0.9115671300423678}, "turn": {}}
Retrieval log: /kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session/longmemeval_s_cleaned_sample20_seed7.json_retrievallog_session_flat-bm25
Exists: True | Size MB: 11.67


In [10]:
run_cmd([sys.executable, 'src/evaluation/print_retrieval_metrics.py', str(retrieval_log)], cwd=REPO_DIR, env=env)


$ /usr/bin/python3 src/evaluation/print_retrieval_metrics.py /kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session/longmemeval_s_cleaned_sample20_seed7.json_retrievallog_session_flat-bm25
Session-level metrics:
	recall_all@5 = 0.9474, 	ndcg_any@5 = 0.659, 	recall_all@10 = 1.0, 	ndcg_any@10 = 0.6717
Turn-level metrics:


CompletedProcess(args=['/usr/bin/python3', 'src/evaluation/print_retrieval_metrics.py', '/kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session/longmemeval_s_cleaned_sample20_seed7.json_retrievallog_session_flat-bm25'], returncode=0)

## Step 2: retrieval-augmented generation

This stage reads the retrieval log and calls an OpenAI-compatible chat completions API. The API key is read by `run_generation.py` from `OPENAI_API_KEY`, so it is not printed in notebook output.


In [11]:
if not OPENAI_API_KEY:
    raise RuntimeError('Set Kaggle Secret OPENAI_API_KEY before running generation/evaluation cells.')

run_id = time.strftime('%Y%m%d-%H%M%S')
generation_out_dir = REPO_DIR / 'generation_logs' / f'{RETRIEVER}-{GRANULARITY}' / GEN_MODEL_ALIAS / 'con'
generation_out_dir.mkdir(parents=True, exist_ok=True)

if RETRIEVER == 'oracle':
    retriever_type = f'oracle-{GRANULARITY}'
else:
    retriever_type = f'flat-{GRANULARITY}'

suffix = f'_{run_id}_kaggle'
cmd = [
    sys.executable, 'run_generation.py',
    '--in_file', str(retrieval_log),
    '--out_dir', str(generation_out_dir),
    '--out_file_suffix', suffix,
    '--model_name', GEN_MODEL_NAME,
    '--model_alias', GEN_MODEL_ALIAS,
    '--retriever_type', retriever_type,
    '--merge_key_expansion_into_value', 'none',
    '--topk_context', str(TOPK_CONTEXT),
    '--history_format', HISTORY_FORMAT,
    '--useronly', USERONLY,
    '--cot', 'true',
    '--con', 'false',
]
if OPENAI_BASE_URL:
    cmd.extend(['--openai_base_url', OPENAI_BASE_URL])

run_cmd(cmd, cwd=REPO_DIR / 'src' / 'generation', env=env)

hyp_files = sorted(generation_out_dir.glob(f'*{suffix}'), key=lambda p: p.stat().st_mtime)
if not hyp_files:
    raise FileNotFoundError(f'No generation output found with suffix {suffix}')
hyp_file = hyp_files[-1]
print('Hypothesis file:', hyp_file)
print('Lines:', sum(1 for _ in hyp_file.open(encoding='utf-8')))


$ /usr/bin/python3 run_generation.py --in_file /kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session/longmemeval_s_cleaned_sample20_seed7.json_retrievallog_session_flat-bm25 --out_dir /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con --out_file_suffix _20260527-113756_kaggle --model_name cx/gpt-5.2 --model_alias router-gpt-5.2 --retriever_type flat-session --merge_key_expansion_into_value none --topk_context 50 --history_format json --useronly false --cot true --con false --openai_base_url https://splashed-nastily-stopped.ngrok-free.dev/v1
Namespace(in_file='/kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session/longmemeval_s_cleaned_sample20_seed7.json_retrievallog_session_flat-bm25', out_dir='/kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con', out_file_suffix='_20260527-113756_kaggle', model_name='cx/gpt-5.2', model_alias='router-gpt-5.2', openai_base_url='https://sp

  0%|          | 0/20 [00:00<?, ?it/s]

{"hypothesis": "Step 1 \u2014 Relevant info From The Chat History\n- Session 2: You said you \u201cjust got\u201d:\n  - a 1/72 scale B-29 bomber model kit (bought)\n  - a 1/24 scale \u201969 Camaro kit (bought)\n- Session 33: You said you \u201crecently finished a simple Revell F-15 Eagle kit\u201d (worked on)\n- Session 44: You said you \u201crecently finished a Tamiya 1/48 scale Spitfire Mk.V\u201d (worked on)\n- Session 41: You said you \u201cstarted working on a diorama featuring a 1/16 scale German Tiger I tank\u201d (worked on)\n\nStep 2 \u2014 Reasoning / Counting\n- Kits you bought: B-29 (1) + \u201969 Camaro (1) = 2\n- Kits you worked on (including finished or in-progress): Revell F-15 (1) + Tamiya Spitfire Mk.V (1) + 1/16 Tiger I (1) = 3\n- Total unique model kits mentioned as bought or worked on = 2 + 3 = 5\n\nAnswer\n5 model kits."}
{
    "question_id": "e56a43b9",
    "question": "How much discount will I get on my next purchase at FreshMart?",
    "answer": "$5"
}


  5%|▌         | 1/20 [10:17<3:15:40, 617.91s/it]

{"hypothesis": "### Relevant Information Extracted From The Chat History\n1. The user said: \u201cevery 100 points translate to a $1 discount on my next purchase.\u201d\n2. The user also said: \u201c500 points gets me a $5 discount on my next purchase.\u201d\n3. The user previously stated they \u201cjust reached 500 points\u201d (the minimum for a discount at FreshMart).\n\n### Reasoning\n1. If **100 points = $1**, then **500 points = 500 \u00f7 100 = 5 dollars**.\n2. This matches the user\u2019s own statement that **500 points gets a $5 discount**.\n\n### Answer\nYou will get a **$5 discount** on your next purchase at FreshMart."}
{
    "question_id": "87f22b4a",
    "question": "How much have I made from selling eggs this month?",
    "answer": "$120"
}


...

{"hypothesis": "Step 1: Extract relevant info from the chat history\n- You said: \u201cI spent $75 on groceries at SaveMart last Thursday.\u201d (Session 15)\n- You also said earlier: you have a SaveMart membership and earn \u201c1% cashback on all purchases.\u201d (Session 10)\n\nStep 2: Reason / calculate the cashback\n- Cashback rate = 1% = 0.01\n- Purchase amount = $75\n- Cashback earned = $75 \u00d7 0.01 = $0.75\n\nAnswer: You earned $0.75 cashback at SaveMart last Thursday."}
{
    "question_id": "5831f84d",
    "question": "How many Crash Course videos have I watched in the past few weeks?",
    "answer": "15"
}


 95%|█████████▌| 19/20 [34:18<00:41, 41.85s/it]

{"hypothesis": "Step 1: Extract relevant information from the chat history\n\n- Session 11 (2023/08/11): You said you had \u201calready finished 10 videos in the past few weeks.\u201d\n- Session 11 (later): You said you \u201cjust finished my 12th video.\u201d\n- Session 29 (2023/09/30): You said you had \u201cwatched 15 Crash Course videos in the past few weeks.\u201d\n\nStep 2: Reason to get the answer\n\n- The question asks: \u201cHow many Crash Course videos have I watched in the past few weeks?\u201d\n- The most recent statement that matches \u201cin the past few weeks\u201d is from 2023/09/30, where you said 15.\n- Therefore, based on the latest relevant info, the answer is: 15.\n\nAnswer: 15"}
Total prompt tokens: 2297858
Total completion tokens: 4832


100%|██████████| 20/20 [34:32<00:00, 103.61s/it]


Hypothesis file: /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/longmemeval_s_cleaned_sample20_seed7.json_retrievallog_session_flat-bm25_testlog_top50context_jsonformat_useronlyfalse_20260527-1138_20260527-113756_kaggle
Lines: 20


## Step 3: official QA evaluation

The official evaluator asks a metric LLM whether each generated answer is correct. For paper-comparable judging, use `METRIC_MODEL_SHORT='gpt-4o'` with an OpenAI endpoint/key. For routers, set `METRIC_MODEL_SHORT` to any alias and `METRIC_MODEL_NAME` to the router model name.


In [12]:
cmd = [sys.executable, 'evaluate_qa.py', METRIC_MODEL_SHORT, str(hyp_file), str(work_file)]
run_cmd(cmd, cwd=REPO_DIR / 'src' / 'evaluation', env=env)

eval_file = Path(str(hyp_file) + f'.eval-results-{METRIC_MODEL_SHORT}')
print('Evaluation log:', eval_file)
print('Exists:', eval_file.exists())


$ /usr/bin/python3 evaluate_qa.py router-gpt-5.2 /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/longmemeval_s_cleaned_sample20_seed7.json_retrievallog_session_flat-bm25_testlog_top50context_jsonformat_useronlyfalse_20260527-1138_20260527-113756_kaggle /kaggle/working/LongMemEval-Experiment/data/longmemeval_s_cleaned_sample20_seed7.json


  0%|          | 0/20 [00:00<?, ?it/s]

{
    "question": "How many model kits have I worked on or bought?",
    "answer": "I have worked on or bought five model kits. The scales of the models are: Revell F-15 Eagle (scale not mentioned), Tamiya 1/48 scale Spitfire Mk.V, 1/16 scale German Tiger I tank, 1/72 scale B-29 bomber, and 1/24 scale '69 Camaro.",
    "hypothesis": "Step 1 \u2014 Relevant info From The Chat History\n- Session 2: You said you \u201cjust got\u201d:\n  - a 1/72 scale B-29 bomber model kit (bought)\n  - a 1/24 scale \u201969 Camaro kit (bought)\n- Session 33: You said you \u201crecently finished a simple Revell F-15 Eagle kit\u201d (worked on)\n- Session 44: You said you \u201crecently finished a Tamiya 1/48 scale Spitfire Mk.V\u201d (worked on)\n- Session 41: You said you \u201cstarted working on a diorama featuring a 1/16 scale German Tiger I tank\u201d (worked on)\n\nStep 2 \u2014 Reasoning / Counting\n- Kits you bought: B-29 (1) + \u201969 Camaro (1) = 2\n- Kits you worked on (including finished or in

  5%|▌         | 1/20 [00:03<01:00,  3.21s/it]

{
    "question": "How much discount will I get on my next purchase at FreshMart?",
    "answer": "$5",
    "hypothesis": "### Relevant Information Extracted From The Chat History\n1. The user said: \u201cevery 100 points translate to a $1 discount on my next purchase.\u201d\n2. The user also said: \u201c500 points gets me a $5 discount on my next purchase.\u201d\n3. The user previously stated they \u201cjust reached 500 points\u201d (the minimum for a discount at FreshMart).\n\n### Reasoning\n1. If **100 points = $1**, then **500 points = 500 \u00f7 100 = 5 dollars**.\n2. This matches the user\u2019s own statement that **500 points gets a $5 discount**.\n\n### Answer\nYou will get a **$5 discount** on your next purchase at FreshMart.",
    "autoeval_label": true
}


...

 90%|█████████ | 18/20 [42:18<06:01, 180.97s/it]

{
    "question": "How much cashback did I earn at SaveMart last Thursday?",
    "answer": "$0.75",
    "hypothesis": "Step 1: Extract relevant info from the chat history\n- You said: \u201cI spent $75 on groceries at SaveMart last Thursday.\u201d (Session 15)\n- You also said earlier: you have a SaveMart membership and earn \u201c1% cashback on all purchases.\u201d (Session 10)\n\nStep 2: Reason / calculate the cashback\n- Cashback rate = 1% = 0.01\n- Purchase amount = $75\n- Cashback earned = $75 \u00d7 0.01 = $0.75\n\nAnswer: You earned $0.75 cashback at SaveMart last Thursday.",
    "autoeval_label": true
}


 95%|█████████▌| 19/20 [42:21<02:07, 127.31s/it]

{
    "question": "How many Crash Course videos have I watched in the past few weeks?",
    "answer": "15",
    "hypothesis": "Step 1: Extract relevant information from the chat history\n\n- Session 11 (2023/08/11): You said you had \u201calready finished 10 videos in the past few weeks.\u201d\n- Session 11 (later): You said you \u201cjust finished my 12th video.\u201d\n- Session 29 (2023/09/30): You said you had \u201cwatched 15 Crash Course videos in the past few weeks.\u201d\n\nStep 2: Reason to get the answer\n\n- The question asks: \u201cHow many Crash Course videos have I watched in the past few weeks?\u201d\n- The most recent statement that matches \u201cin the past few weeks\u201d is from 2023/09/30, where you said 15.\n- Therefore, based on the latest relevant info, the answer is: 15.\n\nAnswer: 15",
    "autoeval_label": true
}
Accuracy: 0.95
	multi-session: 0.875 (8)
	single-session-user: 1.0 (1)
	knowledge-update: 1.0 (3)
	temporal-reasoning: 1.0 (3)
	single-session-assista

100%|██████████| 20/20 [42:23<00:00, 127.18s/it]


## Aggregate QA and retrieval metrics

The QA summary supports any evaluator alias. Retrieval metrics below match the reporting rule in `run_retrieval.py`: skip abstention items and items without user-side target labels.


In [13]:
eval_rows = [json.loads(line) for line in eval_file.read_text(encoding='utf-8').splitlines() if line.strip()]
ref_rows = {row['question_id']: row for row in json.loads(work_file.read_text(encoding='utf-8'))}
retrieval_rows = [json.loads(line) for line in retrieval_log.read_text(encoding='utf-8').splitlines() if line.strip()]
retrieval_by_id = {row['question_id']: row for row in retrieval_rows}

def has_user_side_target(row):
    return any(
        ('has_answer' in turn) and bool(turn['has_answer'])
        for session in row.get('haystack_sessions', [])
        for turn in session
        if turn.get('role') == 'user'
    )

type_to_scores = {}
abstention_scores = []
question_records = []
for row in eval_rows:
    qid = row['question_id']
    ref = ref_rows[qid]
    score = 1 if row['autoeval_label']['label'] else 0
    qtype = ref['question_type']
    type_to_scores.setdefault(qtype, []).append(score)
    if '_abs' in qid:
        abstention_scores.append(score)
    rmetrics = retrieval_by_id.get(qid, {}).get('retrieval_results', {}).get('metrics', {}).get('session', {})
    question_records.append({
        'question_id': qid,
        'question_type': qtype,
        'abstention': '_abs' in qid,
        'correct': bool(score),
        'session_recall_all@5': rmetrics.get('recall_all@5'),
        'session_ndcg_any@5': rmetrics.get('ndcg_any@5'),
        'question': ref['question'],
        'answer': ref['answer'],
        'hypothesis': row.get('hypothesis', ''),
    })

all_scores = [s for scores in type_to_scores.values() for s in scores]
task_scores = [sum(scores) / len(scores) for scores in type_to_scores.values() if scores]

print('Evaluation model:', eval_rows[0]['autoeval_label']['model'] if eval_rows else None)
print('Overall accuracy:', round(sum(all_scores) / len(all_scores), 4) if all_scores else None)
print('Task-averaged accuracy:', round(sum(task_scores) / len(task_scores), 4) if task_scores else None)
print('Abstention accuracy:', round(sum(abstention_scores) / len(abstention_scores), 4) if abstention_scores else None, f'({len(abstention_scores)})')
print()
print('By question type:')
for qtype, scores in sorted(type_to_scores.items()):
    print(f'  {qtype}: {sum(scores) / len(scores):.4f} ({len(scores)})')

retrieval_metric_records = []
for granularity in ['session', 'turn']:
    metric_names = sorted({
        name
        for row in retrieval_rows
        for name in row.get('retrieval_results', {}).get('metrics', {}).get(granularity, {}).keys()
    })
    for metric in metric_names:
        values = []
        for row in retrieval_rows:
            if '_abs' in row['question_id'] or not has_user_side_target(row):
                continue
            value = row.get('retrieval_results', {}).get('metrics', {}).get(granularity, {}).get(metric)
            if value is not None and not (isinstance(value, float) and math.isnan(value)):
                values.append(value)
        if values:
            retrieval_metric_records.append({
                'granularity': granularity,
                'metric': metric,
                'mean': sum(values) / len(values),
                'n': len(values),
            })


Evaluation model: cx/gpt-5.2
Overall accuracy: 0.95
Task-averaged accuracy: 0.975
Abstention accuracy: 1.0 (1)

By question type:
  knowledge-update: 1.0000 (3)
  multi-session: 0.8750 (8)
  single-session-assistant: 1.0000 (5)
  single-session-user: 1.0000 (1)
  temporal-reasoning: 1.0000 (3)


In [15]:
from pathlib import Path
import json
import math

repo_dir = Path('/kaggle/working/LongMemEval-Experiment')

def latest_file(pattern):
    files = [p for p in repo_dir.glob(pattern) if p.is_file()]
    if not files:
        raise FileNotFoundError(pattern)
    return max(files, key=lambda p: p.stat().st_mtime)

eval_file = latest_file('generation_logs/**/*.eval-results-*')
retrieval_log = latest_file('retrieval_logs/**/*_retrievallog_*')
hyp_file = latest_file('generation_logs/**/*_kaggle')

eval_rows = [json.loads(line) for line in eval_file.read_text(encoding='utf-8').splitlines() if line.strip()]
retrieval_rows = [json.loads(line) for line in retrieval_log.read_text(encoding='utf-8').splitlines() if line.strip()]
eval_qids = {row['question_id'] for row in eval_rows}

def load_reference_rows(qids):
    for path in sorted((repo_dir / 'data').glob('*.json'), key=lambda p: -p.stat().st_mtime):
        try:
            rows = json.loads(path.read_text(encoding='utf-8'))
        except Exception:
            continue
        ids = {row.get('question_id') for row in rows if isinstance(row, dict)}
        if qids.issubset(ids):
            return path, rows
    raise FileNotFoundError('No reference JSON contains all evaluated question_id values.')

ref_file, ref_rows = load_reference_rows(eval_qids)
ref_by_id = {row['question_id']: row for row in ref_rows}
retrieval_by_id = {row['question_id']: row for row in retrieval_rows}

def has_user_side_target(row):
    return any(
        ('has_answer' in turn) and bool(turn['has_answer'])
        for session in row.get('haystack_sessions', [])
        for turn in session
        if turn.get('role') == 'user'
    )

type_to_scores = {}
abstention_scores = []
question_records = []

for row in eval_rows:
    qid = row['question_id']
    ref = ref_by_id[qid]
    correct = bool(row['autoeval_label']['label'])
    qtype = ref['question_type']
    type_to_scores.setdefault(qtype, []).append(1 if correct else 0)
    if '_abs' in qid:
        abstention_scores.append(1 if correct else 0)

    rmetrics = retrieval_by_id.get(qid, {}).get('retrieval_results', {}).get('metrics', {}).get('session', {})
    question_records.append({
        'question_id': qid,
        'question_type': qtype,
        'abstention': '_abs' in qid,
        'correct': correct,
        'session_recall_all@5': rmetrics.get('recall_all@5'),
        'session_ndcg_any@5': rmetrics.get('ndcg_any@5'),
        'question': ref['question'],
        'answer': ref['answer'],
        'hypothesis': row.get('hypothesis', ''),
    })

all_scores = [s for scores in type_to_scores.values() for s in scores]
task_scores = [sum(scores) / len(scores) for scores in type_to_scores.values() if scores]

print('Artifacts:')
print('  ref_file     =', ref_file)
print('  retrieval_log=', retrieval_log)
print('  hyp_file     =', hyp_file)
print('  eval_file    =', eval_file)

print('\nQA metrics:')
print('  evaluation_model      =', eval_rows[0]['autoeval_label']['model'] if eval_rows else None)
print('  examples_evaluated    =', len(question_records))
print('  overall_accuracy      =', round(sum(all_scores) / len(all_scores), 4) if all_scores else None)
print('  task_averaged_accuracy=', round(sum(task_scores) / len(task_scores), 4) if task_scores else None)
print('  abstention_accuracy   =', round(sum(abstention_scores) / len(abstention_scores), 4) if abstention_scores else None, f'({len(abstention_scores)})')
print('  num_failures          =', sum(1 for row in question_records if not row['correct']))

print('\nBy question type:')
for qtype, scores in sorted(type_to_scores.items()):
    print(f'  {qtype}: {sum(scores) / len(scores):.4f} ({len(scores)})')

print('\nRetrieval metrics, matching run_retrieval.py reporting rule:')
for granularity in ['session', 'turn']:
    metric_names = sorted({
        name
        for row in retrieval_rows
        for name in row.get('retrieval_results', {}).get('metrics', {}).get(granularity, {}).keys()
    })
    for metric in metric_names:
        values = []
        for row in retrieval_rows:
            if '_abs' in row['question_id'] or not has_user_side_target(row):
                continue
            value = row.get('retrieval_results', {}).get('metrics', {}).get(granularity, {}).get(metric)
            if value is not None and not (isinstance(value, float) and math.isnan(value)):
                values.append(value)
        if values:
            print(f'  {granularity}.{metric}: {sum(values) / len(values):.4f} ({len(values)})')

failed_rows = [row for row in question_records if not row['correct']]
print('\nFailed examples:')
if not failed_rows:
    print('  None')
else:
    for row in failed_rows:
        preview = row['hypothesis'].replace('\n', ' ')[:500]
        print(f"\n  {row['question_id']} | {row['question_type']}")
        print('  question:', row['question'])
        print('  answer:', row['answer'])
        print('  hypothesis_preview:', preview)

Artifacts:
  ref_file     = /kaggle/working/LongMemEval-Experiment/data/longmemeval_s_cleaned_sample20_seed7.json
  retrieval_log= /kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session/longmemeval_s_cleaned_sample20_seed7.json_retrievallog_session_flat-bm25
  hyp_file     = /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/longmemeval_s_cleaned_sample20_seed7.json_retrievallog_session_flat-bm25_testlog_top50context_jsonformat_useronlyfalse_20260527-1138_20260527-113756_kaggle
  eval_file    = /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/longmemeval_s_cleaned_sample20_seed7.json_retrievallog_session_flat-bm25_testlog_top50context_jsonformat_useronlyfalse_20260527-1138_20260527-113756_kaggle.eval-results-router-gpt-5.2

QA metrics:
  evaluation_model      = cx/gpt-5.2
  examples_evaluated    = 20
  overall_accuracy      = 0.95
  task_averaged_accuracy= 0.975
  abstention_accuracy   

In [17]:
overview_rows = sorted(
    question_records,
    key=lambda row: (row['correct'], row['question_type'], row['question_id'])
)

for idx, row in enumerate(overview_rows, start=1):
    preview = row.get('hypothesis', '').replace('\n', ' ')[:220]
    print(f"\n[{idx}] correct={'yes' if row['correct'] else 'no'} | type={row['question_type']} | abstention={row['abstention']}")
    print(f"question_id: {row['question_id']}")
    print(f"session_recall_all@5: {row.get('session_recall_all@5')} | session_ndcg_any@5: {row.get('session_ndcg_any@5')}")
    print(f"question: {row['question']}")
    print(f"answer: {row['answer']}")
    print(f"hypothesis_preview: {preview}")


[1] correct=no | type=multi-session | abstention=False
question_id: 09ba9854
session_recall_all@5: 1.0 | session_ndcg_any@5: 1.0
question: How much will I save by taking the train from the airport to my hotel instead of a taxi?
answer: $50
hypothesis_preview: Step 1 — Extract relevant info from the chat history  - You’re flying into Narita International Airport (NRT) and staying in Shinjuku (Session 14). - Estimated taxi cost from Narita Airport to Tokyo:   - Daytime: JPY 15,

[2] correct=yes | type=knowledge-update | abstention=False
question_id: 5831f84d
session_recall_all@5: 1.0 | session_ndcg_any@5: 1.0
question: How many Crash Course videos have I watched in the past few weeks?
answer: 15
hypothesis_preview: Step 1: Extract relevant information from the chat history  - Session 11 (2023/08/11): You said you had “already finished 10 videos in the past few weeks.” - Session 11 (later): You said you “just finished my 12th video.

[3] correct=yes | type=knowledge-update | abstention=F

In [18]:
failed_rows = [row for row in question_records if not row['correct']]

if not failed_rows:
    print('No failed examples in this evaluation run.')
else:
    for idx, row in enumerate(failed_rows, start=1):
        preview = row.get('hypothesis', '').replace('\n', ' ')[:600]
        print(f"\nFailed example {idx}")
        print(f"question_id: {row['question_id']}")
        print(f"question_type: {row['question_type']}")
        print(f"question: {row['question']}")
        print(f"answer: {row['answer']}")
        print(f"hypothesis_preview: {preview}")


Failed example 1
question_id: 09ba9854
question_type: multi-session
question: How much will I save by taking the train from the airport to my hotel instead of a taxi?
answer: $50
hypothesis_preview: Step 1 — Extract relevant info from the chat history  - You’re flying into Narita International Airport (NRT) and staying in Shinjuku (Session 14). - Estimated taxi cost from Narita Airport to Tokyo:   - Daytime: JPY 15,000–20,000 (about USD $140–$180) (Session 29). - Train costs mentioned:   - Narita Express (N’EX) to Shinjuku: JPY 3,020 (about USD $28) (Session 14).   - Cheapest train option mentioned: JPY 1,000–1,200 (about USD $9–$11) to Shinjuku Station, but not directly to the hotel (Session 14). (You’d still need to walk or pay for local transport.)  Step 2 — Reason to compute savings (


## Optional: long-context baseline

This baseline provides the full recent history to the reader. It is expensive on `longmemeval_s_cleaned.json` and not practical for `longmemeval_m_cleaned.json` with normal Kaggle limits. Enable it only after the smoke test succeeds.


In [20]:
RUN_LONG_CONTEXT_BASELINE = False

if RUN_LONG_CONTEXT_BASELINE:
    full_out_dir = REPO_DIR / 'generation_logs' / 'full-history-session' / GEN_MODEL_ALIAS / 'con'
    full_out_dir.mkdir(parents=True, exist_ok=True)
    full_suffix = f'_{run_id}_kaggle_fullhistory'
    cmd = [
        sys.executable, 'run_generation.py',
        '--in_file', str(work_file),
        '--out_dir', str(full_out_dir),
        '--out_file_suffix', full_suffix,
        '--model_name', GEN_MODEL_NAME,
        '--model_alias', GEN_MODEL_ALIAS,
        '--retriever_type', 'orig-session',
        '--merge_key_expansion_into_value', 'none',
        '--topk_context', '1000',
        '--history_format', HISTORY_FORMAT,
        '--useronly', USERONLY,
        '--cot', 'true',
        '--con', 'false',
    ]
    if OPENAI_BASE_URL:
        cmd.extend(['--openai_base_url', OPENAI_BASE_URL])
    run_cmd(cmd, cwd=REPO_DIR / 'src' / 'generation', env=env)
else:
    print('Skipped. Set RUN_LONG_CONTEXT_BASELINE=True to run this optional baseline.')


Skipped. Set RUN_LONG_CONTEXT_BASELINE=True to run this optional baseline.


## Optional: package outputs

Run this cell to create a downloadable archive under Kaggle output.


In [21]:
archive = ROOT / 'longmemeval_repro_outputs.tar.gz'
run_cmd(['tar', '-czf', str(archive), 'retrieval_logs', 'generation_logs'], cwd=REPO_DIR, env=env, check=False)
print('Archive:', archive)


$ tar -czf /kaggle/working/longmemeval_repro_outputs.tar.gz retrieval_logs generation_logs
Archive: /kaggle/working/longmemeval_repro_outputs.tar.gz
